# Alzheimer's Disease Classification: Model Training & Evaluation

### Overview
Following the Exploratory Data Analysis, this notebook focuses on data preprocessing and building machine learning models.

I will evaluate our models using two distinct datasets:
1.  All Features: Training models on every available feature.
2.  Selected Features: Training models exclusively on the features that demonstrated statistical or strong visual significance during the EDA phase.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings('ignore')

In [6]:
import os

df = pd.read_csv(os.path.join('..', 'data', 'alzheimers_disease_data.csv'))
df.drop(['PatientID', 'DoctorInCharge'], axis=1, inplace=True)
df

,Age,Gender,Ethnicity,EducationLevel,BMI,Smoking,AlcoholConsumption,PhysicalActivity,DietQuality,SleepQuality,...,FunctionalAssessment,MemoryComplaints,BehavioralProblems,ADL,Confusion,Disorientation,PersonalityChanges,DifficultyCompletingTasks,Forgetfulness,Diagnosis
0,73,0,0,2,22.927749,0,13.297218,6.327112,1.347214,9.025679,...,6.518877,0,0,1.725883,0,0,0,1,0,0
1,89,0,0,0,26.827681,0,4.542524,7.619885,0.518767,7.151293,...,7.118696,0,0,2.592424,0,0,0,0,1,0
2,73,0,3,1,17.795882,0,19.555085,7.844988,1.826335,9.673574,...,5.895077,0,0,7.119548,0,1,0,1,0,0
3,74,1,0,1,33.800817,1,12.209266,8.428001,7.435604,8.392554,...,8.965106,0,1,6.481226,0,0,0,0,0,0
4,89,0,0,0,20.716974,0,18.454356,6.310461,0.795498,5.597238,...,6.045039,0,0,0.014691,0,0,1,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2144,61,0,0,1,39.121757,0,1.561126,4.049964,6.555306,7.535540,...,0.238667,0,0,4.492838,1,0,0,0,0,1
2145,75,0,0,2,17.857903,0,18.767261,1.360667,2.904662,8.555256,...,8.687480,0,1,9.204952,0,0,0,0,0,1
2146,77,0,0,1,15.476479,0,4.594670,9.886002,8.120025,5.769464,...,1.972137,0,0,5.036334,0,0,0,0,0,1
2147,78,1,3,1,15.299911,0,8.674505,6.354282,1.263427,8.322874,...,5.173891,0,0,3.785399,0,0,0,0,1,1


In [ ]:
y = df['Diagnosis']
X_all = df.drop('Diagnosis', axis=1)

# based on our statistical and visualization analyses
selected_columns = [
    'FunctionalAssessment', 'ADL', 'MMSE', 'MemoryComplaints', 
    'BehavioralProblems', 'SleepQuality', 'CholesterolHDL', 'CholesterolTriglycerides',
    'Hypertension', 'CardiovascularDisease'
]
X_selected = df[selected_columns]

# Total features available:
X_all.shape[1]

32

In [8]:
RANDOM_STATE = 85

X_train_all, X_test_all, y_train, y_test = train_test_split(
    X_all, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

X_train_selected = X_train_all[selected_columns]
X_test_selected = X_test_all[selected_columns]

#### Checking for Outliers

In [10]:
def check_outliers_iqr(dataframe, features):
    outlier_counts = {}
    for col in features:
        Q1 = dataframe[col].quantile(0.25)
        Q3 = dataframe[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        outliers = dataframe[(dataframe[col] < lower_bound) | (dataframe[col] > upper_bound)]
        if len(outliers) > 0:
            outlier_counts[col] = len(outliers)
            
    return outlier_counts

continuous_features = [
    'Age', 'BMI', 'AlcoholConsumption', 'PhysicalActivity', 'DietQuality', 'SleepQuality',
    'SystolicBP', 'DiastolicBP', 'CholesterolTotal', 'CholesterolLDL', 'CholesterolHDL',
    'CholesterolTriglycerides', 'MMSE', 'FunctionalAssessment', 'ADL'
]

outliers_found = check_outliers_iqr(X_train_all, continuous_features)
outliers_found if outliers_found else "No outliers detected."

'No outliers detected.'

### Feature Encoding

- Binary features: (e.g., `Gender`) are already 0/1.
- Ordinal features: `EducationLevel` has a natural hierarchy (0 to 3), so integer encoding is valid.
- Nominal features `Ethnicity` is the only feature, we will apply a one-hot encoding.

In [ ]:
X_train_all_encoded = pd.get_dummies(X_train_all, columns=['Ethnicity'], dtype=int)
X_test_all_encoded = pd.get_dummies(X_test_all, columns=['Ethnicity'], dtype=int)